# Perdido en el medio: El problema con los contextos largos

"Independientemente de la arquitectura de tu modelo, existe una degradación sustancial del rendimiento cuando incluyes más de 10 documentos recuperados. En resumen: Cuando los modelos deben acceder a información relevante en medio de contextos largos, tienden a ignorar los documentos proporcionados. Ver: https://arxiv.org/abs/2307.03172

Para evitar este problema, puedes reordenar los documentos después de recuperarlos para evitar la degradación del rendimiento."

Por: [Langchain](https://python.langchain.com/docs/modules/data_connection/document_transformers/post_retrieval/long_context_reorder)

![diagrama](../docs/media/diagrams/05.png)

## Librerías

In [2]:
from operator import itemgetter

from dotenv import load_dotenv
from langchain_mistralai import ChatMistralAI
from langchain_mistralai import MistralAIEmbeddings
from langchain_community.document_transformers import LongContextReorder
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

from src.langchain_docs_loader import LangchainDocsLoader, num_tokens_from_string

load_dotenv()

True

## Carga de datos

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=350,
    chunk_overlap=10,
    length_function=num_tokens_from_string,
)

documents = LangchainDocsLoader().load()
documents = text_splitter.split_documents(documents)

## Creación de retriever

In [4]:
retriever = Chroma.from_documents(documents, embedding=MistralAIEmbeddings()).as_retriever(
    search_type="mmr",
    search_kwargs={
        "k": 10,
        "fetch_k": 50,
    },
)

## Consulta con el retriever

In [5]:
relevant_docs = retriever.invoke(
    "How to use LCEL ainvoke with a retriever?"
)
relevant_docs

[Document(id='2b97977d-67ca-4a2e-8745-6e5ae31ffeba', metadata={'title': 'Build a semantic search engine with LangChain - Docs by LangChain', 'language': 'en', 'source': 'https://docs.langchain.com/oss/python/langchain/knowledge-base'}, page_content='## \u200b4. Retrievers\n\nLangChain [VectorStore](https://reference.langchain.com/python/langchain_core/vectorstores/?h=#langchain_core.vectorstores.base.VectorStore) objects do not subclass [Runnable](https://reference.langchain.com/python/langchain_core/runnables/#langchain_core.runnables.Runnable). LangChain [Retrievers](https://reference.langchain.com/python/langchain_core/retrievers/#langchain_core.retrievers.BaseRetriever) are Runnables, so they implement a standard set of methods (e.g., synchronous and asynchronous invoke and batch operations). Although we can construct retrievers from vector stores, retrievers can interface with non-vector store sources of data, as well (such as external APIs).\n\nWe can create a simple version of t

## Reordenado de documentos

In [6]:
reordering = LongContextReorder()
reordered_docs = list(reordering.transform_documents(relevant_docs))
reordered_docs

[Document(id='880877c3-08f5-48c2-a456-a9d32a8a7e41', metadata={'source': 'https://docs.langchain.com/oss/python/langchain/messages', 'language': 'en', 'title': 'Messages - Docs by LangChain'}, page_content='response = model.invoke(messages)\n\n```\n\nAttributes\n\n[\u200b](#param-text)textstringThe text content of the message.\n\n[\u200b](#param-content)contentstring | dict[]The raw content of the message.\n\n[\u200b](#param-content-blocks)content_blocksContentBlock[]The standardized [content blocks](#message-content) of the message.\n\n[\u200b](#param-tool-calls)tool_callsdict[] | NoneThe tool calls made by the model.\n\nEmpty if no tools are called.\n\n[\u200b](#param-id)idstringA unique identifier for the message (either automatically generated by LangChain or returned in the provider response)\n\n[\u200b](#param-usage-metadata)usage_metadatadict | NoneThe usage metadata of the message, which can contain token counts when available.\n\n[\u200b](#param-response-metadata)response_meta

## Uso del reordenador en nuestro pipeline de `Retrieval Augmented Generation`

In [7]:
def combine_documents(documents: list[Document]) -> str:
    return "\n\n".join([doc.page_content for doc in documents])


prompt = PromptTemplate.from_template(
    """Given the following text extracts:
-----
{context}
-----
                                      
Answer the following question, if you don't know the answer, just write "I don't know.

Question: {question}"""
)

llm = ChatMistralAI(temperature=0)

stuff_chain = (
    {
        "context": itemgetter("question")
        | retriever
        | reordering.transform_documents
        | combine_documents,
        "question": itemgetter("question"),
    }
    | prompt
    | llm
)

In [8]:
response = stuff_chain.invoke(input={"question": "How to create a chain using LCEL?"}).content
print(response)

# Creating a Chain Using LCEL (LangChain Expression Language)

To create a chain using LangChain Expression Language (LCEL), you'll need to define a sequence of operations using LangChain's expression syntax. Here's how to do it:

## Basic LCEL Chain Example

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import OpenAI

# Define the prompt template
prompt = ChatPromptTemplate.from_template("Tell me a joke about {topic}")

# Define the output parser
output_parser = StrOutputParser()

# Create the chain using LCEL
chain = prompt | OpenAI(temperature=0.9) | output_parser

# Invoke the chain
result = chain.invoke({"topic": "cats"})
print(result)
```

## More Complex Example with Multiple Steps

```python
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_community.llms import OpenAI
from langchain_community.tool